# 🚀 SpaceX Falcon 9 – Notebook 8: Predictive Analysis (Machine Learning)

Build and evaluate classification models to predict Falcon 9 first stage landing success.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import preprocessing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

df = pd.read_csv('dataset_part_2.csv')
print("Shape:", df.shape)
df.head()

## Feature Engineering

In [ ]:
features = df[['FlightNumber','PayloadMass','Flights','Block',
               'ReusedCount','Orbit','LaunchSite','GridFins','Reused','Legs','LandingPad']]

# One-hot encode categoricals
features_one_hot = pd.get_dummies(features, columns=['Orbit','LaunchSite','LandingPad'])
features_one_hot = features_one_hot.fillna(0)
print("Features shape:", features_one_hot.shape)
features_one_hot.head()

In [ ]:
# Scale features
scaler = preprocessing.StandardScaler()
X = scaler.fit_transform(features_one_hot)
Y = df['Class'].values

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=2)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## Model 1 – Logistic Regression

In [ ]:
params_lr = {'C': [0.01, 0.1, 1, 10, 100], 'solver': ['lbfgs']}
lr = GridSearchCV(LogisticRegression(max_iter=10000), params_lr, cv=10)
lr.fit(X_train, Y_train)
print("Best params:", lr.best_params_)
print("Test Accuracy:", lr.score(X_test, Y_test).round(4))

## Model 2 – Support Vector Machine

In [ ]:
params_svm = {'C': [1, 4, 8, 16, 32, 64], 'kernel': ['linear', 'rbf', 'sigmoid'],
              'gamma': [1e-3, 1e-4]}
svm = GridSearchCV(SVC(), params_svm, cv=10)
svm.fit(X_train, Y_train)
print("Best params:", svm.best_params_)
print("Test Accuracy:", svm.score(X_test, Y_test).round(4))

## Model 3 – Decision Tree

In [ ]:
params_dt = {'criterion': ['gini','entropy'],
             'max_depth': [2,4,6,8,10],
             'min_samples_split': [2,5,10]}
dt = GridSearchCV(DecisionTreeClassifier(), params_dt, cv=10)
dt.fit(X_train, Y_train)
print("Best params:", dt.best_params_)
print("Test Accuracy:", dt.score(X_test, Y_test).round(4))

## Model 4 – K-Nearest Neighbors

In [ ]:
params_knn = {'n_neighbors': [2,3,4,5,6,7,8,9,10],
              'algorithm': ['auto','ball_tree','kd_tree','brute'],
              'p': [1,2]}
knn = GridSearchCV(KNeighborsClassifier(), params_knn, cv=10)
knn.fit(X_train, Y_train)
print("Best params:", knn.best_params_)
print("Test Accuracy:", knn.score(X_test, Y_test).round(4))

## Confusion Matrix – Best Model

In [ ]:
def plot_confusion_matrix(y_true, y_pred, title='Confusion Matrix'):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Fail','Success'], yticklabels=['Fail','Success'])
    plt.title(title)
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.tight_layout()
    plt.savefig(f'confusion_{title.replace(" ","_")}.png', dpi=150)
    plt.show()

# Evaluate all models
models = {'Logistic Regression': lr, 'SVM': svm, 'Decision Tree': dt, 'KNN': knn}
results = {}
for name, model in models.items():
    acc = model.score(X_test, Y_test)
    results[name] = acc
    print(f"{name}: {acc:.4f}")

best_name = max(results, key=results.get)
best_model = models[best_name]
Y_pred = best_model.predict(X_test)
plot_confusion_matrix(Y_test, Y_pred, title=best_name)
print(f"\n🏆 Best Model: {best_name} ({results[best_name]:.4f})")

## Model Comparison

In [ ]:
plt.figure(figsize=(8,4))
plt.bar(results.keys(), results.values(), color=['steelblue','orange','green','red'])
plt.title('Model Accuracy Comparison')
plt.ylabel('Accuracy')
plt.ylim(0.5, 1.0)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150)
plt.show()